# Portfolio Code 

In [ ]:
# Libraries 

import json
import os
import pathlib

import earthpy
import hvplot.xarray
import rioxarray as rxr
import xarray as xr

import geopandas as gpd
import hvplot.pandas

from glob import glob
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Gila River Valley

project = earthpy.Project("Gila River Vegetation", dirname='gila_river_veg')
project.get_data()

# Load in the boundary data
aitsn_gdf = gpd.read_file(project.project_dir / 'tl_2020_us_aitsn')
# Check that it worked
aitsn_gdf

# Change data type 
aitsn_gdf['AIANNHCE'] = aitsn_gdf['AIANNHCE'].astype(int)

# Select Gila River Valley Reservation
Gila_boundary_gdf = aitsn_gdf.loc[aitsn_gdf.AIANNHCE==1310].dissolve()
# Plot the results with web tile images
Gila_boundary_gdf.hvplot(
    geo=True, tiles='EsriImagery',
    fill_color=None, line_color='black',
    line_width = 2,
    title='Gila River Subdivision',
    frame_width=500)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [ ]:
# NDVI 
project = earthpy.Project("Gila River Vegetation", dirname='gila_river_veg')

# Get a sorted list of NDVI tif file paths
ndvi_paths = sorted(list(project.project_dir.rglob('MOD13Q1.061__250m_16_days_NDVI*.tif')))

# Display the first and last three files paths to check the pattern
ndvi_paths[:3], ndvi_paths[-3:]


ndvi_das = []

for ndvi_path in ndvi_paths:
    # Get date from file name
    fname = ndvi_path.name  # just the filename, not the whole path
    # find the substring "doy" and take the next 7 digits (YYYYDDD)
    doy_index = fname.find("doy") + 3
    date_str = fname[doy_index:doy_index + 7]  # e.g. '2001145'
    year = int(date_str[:4])
    doy = int(date_str[4:])
    date = pd.to_datetime(f"{year}-{doy}", format="%Y-%j")
    
    # Open dataset
    da = rxr.open_rasterio(ndvi_path, masked=True).squeeze()
    
    # Add date dimension and clean up metadata 
    da = da.assign_coords({'date': date}).expand_dims({'date': [date]})
    da.name = 'NDVI'
    
    # Prepare for concatenation
    ndvi_das.append(da)

ndvi_da = xr.combine_by_coords(ndvi_das, coords=['date'])


In [ ]:
# 2001–2011 
ndvi_2001_2011 = ndvi_da.sel(date=slice('2001', '2011'))

# temporal mean
mean_2001_2011 = ndvi_2001_2011.mean(dim='date')

# 2012–2022 
ndvi_2012_2022 = ndvi_da.sel(date=slice('2012', '2022'))

# temporal mean 
mean_2012_2022 = ndvi_2012_2022.mean(dim='date')

# Compute the difference in NDVI before and after
ndvi_diff = mean_2012_2022 - mean_2001_2011

# save as data array
ndvi_diff_da = ndvi_diff['NDVI']

In [ ]:
# scale NDVI first 
ndvi_diff_scaled = ndvi_diff_da * 0.0001

(
    ndvi_diff_scaled.hvplot(x='x', y='y', cmap='PiYG', geo=True, 
                            title='Difference in NVDI from the 2000s to 2010s', ylabel='Latitude', xlabel='Longitude'
    *
    Gila_boundary_gdf.hvplot(geo=True, fill_color=None, line_color='black') # add boundary
)